# NeuroGolf 2026 — Reverse-Engineering ALL 400 tasks

For **every** task this notebook lets you see, in one place:
- the **task** (input→output grid plots),
- the **baseline ONNX graph** (hoangvux's solution) drawn as a DAG (or op-histogram if huge),
- its **op breakdown, node count, cost & points**,
- the **ground-truth rule** (from the public ARC-DSL reference — the program that defines the transform).

**Why:** the baseline graph is a *guaranteed-correct rule oracle*. Instead of guessing rules from
examples (which fails on arc-gen), we read the true rule here. Cross-referencing **cost** with **rule
simplicity** tells us which tasks hoangvux implemented *wastefully* — those are the ones we can rewrite
**cheaper** and bank points (the task135 pattern).

> This is an automated *catalog* of all 400. Deep hand-rewrites are done per-task in `from_scratch.ipynb`.
> Requires onnxruntime>=1.27 (see the venv). Run from the project root.

## Setup — run first

In [ ]:
import os, sys, math, json, re, copy
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from collections import defaultdict, Counter
import onnx, onnxruntime
from onnx import numpy_helper

TASK_DIR="data"; BASE_DIR="baseline_v22"; OUT_DIR="repairs"; REF="arc_dsl_ref"
_ARC=[(0,0,0),(30,147,255),(250,61,49),(78,204,48),(255,221,0),(153,153,153),(229,59,163),(255,133,28),(136,216,241),(147,17,49)]
PAL=ListedColormap([tuple(c/255 for c in rgb) for rgb in _ARC])
def load_task(t): return json.load(open(f"{TASK_DIR}/task{t:03d}.json"))

# DSL reference (ground-truth rule per task)
MAP={int(k):v for k,v in json.load(open(f"{REF}/task_to_hash.json")).items()}
_src=open(f"{REF}/solvers.py").read()
SOLVERS={m.group(1):m.group(2).rstrip() for m in re.finditer(r'def solve_([0-9a-f]{8})\(I\):\n(.*?)(?=\ndef |\Z)',_src,re.DOTALL)}
def dsl_rule(t): return SOLVERS.get(MAP.get(t,''),'(no reference)')

CATALOG=pd.read_csv(f"{OUT_DIR}/catalog.csv")
print("catalog rows:", len(CATALOG))

## The full 400-task catalog table
Sort/filter this however you like (it's a pandas DataFrame).

In [ ]:
pd.set_option("display.max_rows", 400); pd.set_option("display.max_colwidth", 70)
CATALOG[["task","hash","status","n_nodes","cost","points","ops"]]

## Graph drawer + per-task deep-dive `analyze(t)`

In [ ]:
def draw_onnx(path_or_model, ax=None, title=None, max_nodes=45):
    m=onnx.load(path_or_model) if isinstance(path_or_model,str) else path_or_model
    nodes=list(m.graph.node); own=ax is None
    if own: fig,ax=plt.subplots(figsize=(11,5))
    if len(nodes)>max_nodes:
        c=Counter(n.op_type for n in nodes).most_common()
        ax.barh([k for k,_ in c][::-1],[v for _,v in c][::-1],color="#8e44ad")
        ax.set_title(f"{title or ''} ({len(nodes)} nodes — op counts)",fontsize=9); ax.tick_params(labelsize=7)
        if own: plt.tight_layout(); plt.show()
        return
    prod={o:i for i,n in enumerate(nodes) for o in n.output}
    preds={i:[prod[x] for x in n.input if x in prod] for i,n in enumerate(nodes)}
    layer={}
    def L(i):
        if i not in layer: layer[i]=0 if not preds[i] else max(L(p) for p in preds[i])+1
        return layer[i]
    for i in range(len(nodes)): L(i)
    byl=defaultdict(list)
    for i,l in layer.items(): byl[l].append(i)
    pos={}
    for l,ids in byl.items():
        for k,i in enumerate(ids): pos[i]=(l,-(k-(len(ids)-1)/2))
    for i in range(len(nodes)):
        x2,y2=pos[i]
        for p in preds[i]:
            x1,y1=pos[p]; ax.annotate("",xy=(x2,y2),xytext=(x1,y1),arrowprops=dict(arrowstyle="->",color="#bbb",lw=0.7))
    for i,n in enumerate(nodes):
        x,y=pos[i]; ax.text(x,y,n.op_type,ha="center",va="center",fontsize=6.5,bbox=dict(boxstyle="round,pad=0.3",fc="#dceefb",ec="#2980b9",lw=0.8))
    ax.set_title(f"{title or 'graph'} ({len(nodes)} nodes)",fontsize=9); ax.axis("off"); ax.set_xlim(-0.6,max(byl)+0.6)
    if own: plt.tight_layout(); plt.show()

def analyze(t, n_ex=3):
    """Full reverse-engineering view of one task: grids, graph, ops, cost, rule."""
    row=CATALOG[CATALOG.task==t].iloc[0]
    print(f"===== task{t:03d}  (ARC hash {row['hash']}, status={row['status']}) =====")
    print(f"nodes={row['n_nodes']}  cost={row['cost']}  points={row['points']}")
    print(f"ops: {row['ops']}")
    print("\nGROUND-TRUTH RULE (ARC-DSL):")
    print(dsl_rule(t))
    # grids
    d=load_task(t); pairs=d["train"][:n_ex]+d["test"][:1]
    fig,ax=plt.subplots(2,len(pairs),figsize=(3*len(pairs),6))
    if len(pairs)==1: ax=ax.reshape(2,1)
    for k,ex in enumerate(pairs):
        ax[0,k].imshow(np.array(ex["input"]),cmap=PAL,vmin=0,vmax=9); ax[0,k].set_title("in"); ax[0,k].axis("off")
        ax[1,k].imshow(np.array(ex["output"]),cmap=PAL,vmin=0,vmax=9); ax[1,k].set_title("out"); ax[1,k].axis("off")
    plt.suptitle(f"task{t:03d} grids"); plt.tight_layout(); plt.show()
    # graph
    draw_onnx(f"{BASE_DIR}/task{t:03d}.onnx", title=f"task{t:03d} baseline graph")

analyze(1)

## Deep-dive any task — just change the number

In [ ]:
analyze(135)   # example: the trivial-crop task we golfed 43406 -> 374

In [ ]:
analyze(7)     # example: already-minimal (single Einsum) — not worth golfing

## Golf-target finder
Tasks that are **expensive** but have a **short DSL rule** (few ops) are the best rewrite candidates —
that's the task135 pattern (a big graph implementing a simple rule). This ranks them.

In [ ]:
cat=CATALOG.copy()
cat["cost_num"]=pd.to_numeric(cat["cost"],errors="coerce")
cat["dsl_len"]=cat["task"].map(lambda t: len([l for l in dsl_rule(t).split("\n") if l.strip() and "return" not in l]))
cand=cat[(cat.status=="baseline") & (cat.cost_num>3000)].copy()
cand=cand.sort_values(["dsl_len","cost_num"],ascending=[True,False])
print("Best golf candidates (short rule, high cost):")
cand[["task","hash","cost","points","n_nodes","dsl_len"]].head(25)

## Export everything (optional)
Write a PNG of every task's baseline graph to `graphs/` (400 files, ~couple minutes).

In [ ]:
def export_all_graphs(folder="graphs"):
    os.makedirs(folder,exist_ok=True)
    for t in range(1,401):
        fig,ax=plt.subplots(figsize=(10,6))
        draw_onnx(f"{BASE_DIR}/task{t:03d}.onnx",ax=ax,title=f"task{t:03d}")
        fig.savefig(f"{folder}/task{t:03d}.png",dpi=70,bbox_inches="tight"); plt.close(fig)
        if t%50==0: print(f"...{t}/400")
    print("done -> graphs/")
# export_all_graphs()   # uncomment to run
print("Call export_all_graphs() to dump all 400 baseline graph PNGs.")